## Install dependencies

In [ ]:
!pip install -q obonet biopython scikit-learn tqdm psutil

In [ ]:
!wget -q https://github.com/bbuchfink/diamond/releases/download/v2.1.10/diamond-linux64.tar.gz
!tar -xzf diamond-linux64.tar.gz
!mv diamond ${HOME}/diamond
!chmod +x ${HOME}/diamond
!export PATH="${HOME}:$PATH"
DIAMOND = "/root/diamond"

In [ ]:
!{DIAMOND} version

## Imports + logging utils

In [ ]:
import os, sys, time, gc, psutil, subprocess
from contextlib import contextmanager
from collections import defaultdict

import numpy as np
import pandas as pd
import obonet
from Bio import SeqIO
from functools import lru_cache
from tqdm.auto import tqdm
from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
input_base_path = "/kaggle/input/cafa-6-protein-function-prediction"
work_dir = "/kaggle/working"
os.makedirs(work_dir, exist_ok=True)

def log(msg):
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}", flush=True)

@contextmanager
def timed(name):
    t0 = time.time()
    log(f"START: {name}")
    try:
        yield
    finally:
        dt = time.time() - t0
        log(f"END:   {name} | {dt/60:.2f} min ({dt:.1f} sec)")

def log_ram(note=""):
    rss = psutil.Process(os.getpid()).memory_info().rss / (1024**3)
    log(f"RAM RSS {note}: {rss:.2f} GB")

## Load train terms + taxonomy + select top taxons

In [ ]:
with timed("Load train terms + taxonomy + filter taxons"):
    train_terms_df = pd.read_csv(f"{input_base_path}/Train/train_terms.tsv", sep="\t")
    train_taxon_df = pd.read_csv(f"{input_base_path}/Train/train_taxonomy.tsv", sep="\t", header=None)
    train_taxon_df.columns = ["ProtID", "Taxon"]

    train_taxon_map = train_taxon_df.set_index("ProtID")["Taxon"].astype(str).to_dict()
    train_terms_df["Taxon"] = train_terms_df["EntryID"].map(train_taxon_map).astype(str)

    cdf = train_terms_df["Taxon"].value_counts().cumsum() / len(train_terms_df)
    selected_train_taxon = cdf[cdf <= 0.95].index.tolist()

    train_terms_df = train_terms_df[train_terms_df["Taxon"].isin(selected_train_taxon)].reset_index(drop=True)

log(f"Filtered train_terms_df shape: {train_terms_df.shape}")
log(f"Selected taxons: {len(selected_train_taxon)}")

## Load ontology + propagated/pruned records

In [ ]:
ont_graph = obonet.read_obo(f"{input_base_path}/Train/go-basic.obo")
parents_map = {n: list(ont_graph.successors(n)) for n in ont_graph.nodes()}

@lru_cache(maxsize=None)
def ancestors_of(term: str) -> frozenset:
    out = set()
    stack = list(parents_map.get(term, []))
    while stack:
        p = stack.pop()
        if p in out:
            continue
        out.add(p)
        stack.extend(parents_map.get(p, []))
    return frozenset(out)

def propagated_terms(term_list):
    s = set(term_list)
    anc = set()
    for t in s:
        anc.update(ancestors_of(t))
    return s | anc

def pruned_leaf_terms(term_list):
    s = set(term_list)
    removable = set()
    for t in s:
        removable.update(ancestors_of(t))
    return s - removable

term_ns_map = {k: v.get("namespace") for k, v in ont_graph.nodes(data=True)}
aspect_ns_map = {
    "biological_process": "P",
    "cellular_component": "C",
    "molecular_function": "F",
}

## Build propagated/pruned dataframes

In [ ]:
with timed("Build propagated/pruned dataframes"):
    grouped_terms = (
        train_terms_df.groupby("EntryID", as_index=False)
        .agg(term_list=("term", list), Taxon=("Taxon", "first"))
    )

    grouped_terms["prop_terms"] = grouped_terms["term_list"].map(propagated_terms)
    grouped_terms["pruned_terms"] = grouped_terms["term_list"].map(pruned_leaf_terms)

    train_terms_df_propagated = (
        grouped_terms[["EntryID", "Taxon", "prop_terms"]]
        .rename(columns={"prop_terms": "term"})
        .explode("term")
    )
    train_terms_df_pruned = (
        grouped_terms[["EntryID", "Taxon", "pruned_terms"]]
        .rename(columns={"pruned_terms": "term"})
        .explode("term")
    )

    train_terms_df_propagated["aspect"] = (
        train_terms_df_propagated["term"].map(term_ns_map).map(aspect_ns_map)
    )
    train_terms_df_pruned["aspect"] = (
        train_terms_df_pruned["term"].map(term_ns_map).map(aspect_ns_map)
    )

log(f"Propagated rows: {len(train_terms_df_propagated)} | Pruned rows: {len(train_terms_df_pruned)}")

## Specificity + filter GO terms

In [ ]:
with timed("Compute specificity + select GO terms"):
    pruned_cnt = train_terms_df_pruned.groupby(["aspect", "term"], as_index=False).agg(
        pruned_count=("EntryID", "count")
    )
    prop_cnt = train_terms_df_propagated.groupby(["aspect", "term"], as_index=False).agg(
        prop_count=("EntryID", "count")
    )
    goterm_count_df = prop_cnt.merge(pruned_cnt, how="left", on=["aspect", "term"]).fillna({"pruned_count": 0})
    goterm_count_df["specificity"] = goterm_count_df["pruned_count"] / (goterm_count_df["prop_count"] + 1e-12)

    MSL = 0.2
    MSS_BP = goterm_count_df.loc[goterm_count_df["aspect"] == "P", "pruned_count"].quantile(0.8)
    MSS_CC = goterm_count_df.loc[goterm_count_df["aspect"] == "C", "pruned_count"].quantile(0.8)
    MSS_MF = goterm_count_df.loc[goterm_count_df["aspect"] == "F", "pruned_count"].quantile(0.8)

    selected_go_terms = goterm_count_df.loc[
        (
            ((goterm_count_df["pruned_count"] >= MSS_BP) & (goterm_count_df["aspect"] == "P"))
            | ((goterm_count_df["pruned_count"] >= MSS_CC) & (goterm_count_df["aspect"] == "C"))
            | ((goterm_count_df["pruned_count"] >= MSS_MF) & (goterm_count_df["aspect"] == "F"))
        )
        & (goterm_count_df["specificity"] >= MSL),
        "term",
    ]

    train_df_final = train_terms_df_pruned[train_terms_df_pruned["term"].isin(selected_go_terms)]

log(f"Selected GO terms: {len(selected_go_terms)}")
log(f"Final pruned coverage: {len(train_df_final) / len(train_terms_df_pruned):.3f}")

## Multi-hot label matrix + train protein order

In [ ]:
with timed("Build label matrix"):
    grouped = train_df_final.groupby("EntryID")["term"].apply(list).reset_index(name="go_terms_list")
    mlb = MultiLabelBinarizer()
    label_matrix = mlb.fit_transform(grouped["go_terms_list"]).astype(np.float32)

    train_entry_ids = grouped["EntryID"].tolist()
    train_id_to_row = {pid: i for i, pid in enumerate(train_entry_ids)}

log(f"Train proteins used: {len(train_entry_ids)} | Labels: {label_matrix.shape[1]}")

## Load sequences + preprocess + write FASTA for DIAMOND DB/query

In [ ]:
standard_aa = set("ACDEFGHIKLMNPQRSTVWY")

def preprocess_sequence(seq: str) -> str:
    seq = seq.strip().upper()
    return "".join(ch if ch in standard_aa else "X" for ch in seq)

def write_fasta(ids, seqs, path):
    with open(path, "w") as f:
        for pid, s in zip(ids, seqs):
            f.write(f">{pid}\n{s}\n")

with timed("Load + preprocess train sequences (DIAMOND)"):
    seq_dict = {}
    for rec in SeqIO.parse(f"{input_base_path}/Train/train_sequences.fasta", "fasta"):
        seq_id = rec.id.split("|")[1]
        seq_dict[seq_id] = preprocess_sequence(str(rec.seq))

    train_seqs = [seq_dict[pid] for pid in train_entry_ids]
    train_fasta_db = f"{work_dir}/train_db.fasta"
    write_fasta(train_entry_ids, train_seqs, train_fasta_db)

with timed("Load + preprocess test sequences (DIAMOND)"):
    test_ids, test_seqs = [], []
    for rec in SeqIO.parse(f"{input_base_path}/Test/testsuperset.fasta", "fasta"):
        taxon = rec.description.split()[-1]
        if taxon not in selected_train_taxon:
            continue
        test_ids.append(rec.id)
        test_seqs.append(preprocess_sequence(str(rec.seq)))

    test_fasta_q = f"{work_dir}/test_query.fasta"
    write_fasta(test_ids, test_seqs, test_fasta_q)

## Build DIAMOND database

In [ ]:
diamond_db = f"{work_dir}/train_db.dmnd"
with timed("DIAMOND makedb"):
    cmd = [
        DIAMOND, "makedb",
        "--in", train_fasta_db,
        "--db", diamond_db,
    ]
    log(" ".join(cmd))
    subprocess.run(cmd, check=True)

## Run DIAMOND blastp to get top-k neighbors per test protein

In [ ]:
k = 50
threads = max(1, os.cpu_count() // 2)
diamond_out = f"{work_dir}/diamond_hits.m8"

with timed("DIAMOND blastp (top-k, with qlen)"):
    cmd = [
        DIAMOND, "blastp",
        "--db", diamond_db,
        "--query", test_fasta_q,
        "--out", diamond_out,
        "--outfmt", "6", "qseqid", "sseqid", "bitscore", "qlen",
        "--max-target-seqs", str(k),
        "--threads", str(threads),
        "--sensitive",
        "--evalue", "1e-1",
        "--quiet",
    ]
    log(" ".join(cmd))
    subprocess.run(cmd, check=True)

log(f"DIAMOND output: {diamond_out}")

## Parse DIAMOND hits → build per-query neighbor lists

In [ ]:
with timed("Parse DIAMOND hits"):
    hits = pd.read_csv(
        diamond_out,
        sep="\t",
        header=None,
        names=["qseqid", "sseqid", "bitscore", "qlen"],
    )

    hits = hits[hits["sseqid"].isin(train_id_to_row)].copy()
    hits["s_row"] = hits["sseqid"].map(train_id_to_row).astype(np.int32)

    hits.sort_values(["qseqid", "bitscore"], ascending=[True, False], inplace=True)

log(f"Hits rows: {len(hits)} | Unique queries with hits: {hits['qseqid'].nunique()}")

## KNN weighted label averaging using DIAMOND scores

In [ ]:
with timed("Predict using DIAMOND-KNN (bitscore / qlen normalized)"):
    n_labels = label_matrix.shape[1]
    prediction_number_limit = 1500
    selection_percentile = np.round(1 - prediction_number_limit / n_labels, 2)

    predictions = {}
    grouped_hits = hits.groupby("qseqid", sort=False)

    for qid in tqdm(test_ids, desc="Predict", mininterval=5):
        if qid not in grouped_hits.groups:
            predictions[qid] = []
            continue

        h = grouped_hits.get_group(qid).head(k)

        rows = h["s_row"].to_numpy()
        bs = h["bitscore"].to_numpy(dtype=np.float32)
        qlen = h["qlen"].iloc[0]  # same for all hits of this query

        bs_norm = bs / (qlen + 1e-8)
        w = bs_norm / (bs_norm.sum() + 1e-8)

        scores = (w[:, None] * label_matrix[rows]).sum(axis=0)
        # TODO: Should we divide this by k?
            # It doesn't change the order.
            # What other difference does it make?

        thr = np.quantile(scores, selection_percentile)
        idx = np.flatnonzero(scores > thr)

        predictions[qid] = [(mlb.classes_[t], float(scores[t])) for t in idx]

log(f"Total predicted proteins: {len(predictions)}")

## Submission

In [ ]:
submission_df = pd.DataFrame(predictions.items()).explode(1)
submission_df[2] = submission_df[1].map(lambda x: x[0] if isinstance(x, tuple) else None)
submission_df[3] = submission_df[1].map(lambda x: x[1] if isinstance(x, tuple) else None)
submission_df = submission_df.drop(columns=1).dropna()
submission_df[3] = submission_df[3].round(3)
display(submission_df.head())

In [ ]:
out_path = f"{work_dir}/submission.tsv"
submission_df.to_csv(out_path, sep="\t", header=None, index=False)
log(f"Wrote: {out_path}")